# GA Forget5 — 학습률 스케줄 수정 비교 실험

이전 FAST GA와 동일한 모델·데이터·seed 233·8 epochs·GPU당 batch 4·accumulation 16을 사용합니다.
수정 사항은 **성공한 optimizer update당 scheduler를 한 번 진행**하는 것과 검증 로그입니다.
`data_module.py`와 GA loss는 변경하지 않습니다. 논문 수치 재현을 보장하는 노트북은 아닙니다.

- 새 Kaggle 노트북, Internet ON, GPU T4 ×2에서 실행하세요. 이전 GA checkpoint를 Input에 연결할 필요가 없습니다.
- 설치 셀 → Python 커널 재시작 셀 → 연결이 돌아온 후 환경 확인 셀부터 순서대로 실행합니다.
  **Stop Session으로 세션을 새로 만들지 마세요.**
- 라이브러리: transformers 4.46.3, tokenizers 0.20.3, peft 0.13.2, accelerate 1.14.0, deepspeed 0.19.7.
- 공개 finetuned 모델에서 새로 시작합니다. 이전 GA step32에 이어서 학습하지 않습니다.
- 새 결과 폴더: `/kaggle/working/models/llava_phi_GA_forget5_fast_scheduler_fix`.
- 매 업데이트에 LR·scheduler index·overflow skip 횟수를 검사합니다. 첫 epoch의 4회 검사를 통과하면 같은 실행을 계속합니다.
- `step_32`는 기존과 동일하게 **업데이트 시도 32회**입니다. overflow가 있으면 실제 성공 횟수는 더 적으며, 이를 별도 기록합니다.
  데이터 노출량을 유지하기 위해 8 epochs 뒤에 추가 학습하지 않습니다.
- effective batch 128은 accumulation을 모두 채운 경우입니다. epoch 마지막 짧은 구간은 기존 동작을 유지합니다.


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d FIUBench_kaggle ]; then
  git clone https://github.com/npngjjm/FIUBench_kaggle.git
fi
python -m pip uninstall -y torchao || true
python -m pip install --no-cache-dir --upgrade \
  'transformers==4.46.3' \
  'tokenizers==0.20.3' \
  'peft==0.13.2' \
  'accelerate==1.14.0' \
  'deepspeed==0.19.7' \
  'huggingface_hub==0.36.2' \
  'numpy==2.1.3' \
  hydra-core omegaconf datasets sentencepiece protobuf scipy pillow \
  pyyaml requests tqdm rouge_score scikit-learn

echo 'INSTALL_COMPLETE — 커널을 한 번 재시작하세요.'


설치 후 아래 셀을 한 번 실행해 Python 커널을 재시작하세요. 연결이 돌아오면 재시작 셀은
다시 실행하지 말고 환경 확인 셀부터 진행합니다.


In [ ]:
import os
os._exit(0)


In [ ]:
import os
from pathlib import Path
from importlib.metadata import version
import torch

repo = Path('/kaggle/working/FIUBench_kaggle')
assert repo.is_dir(), '저장소가 없습니다. clone/설치 셀을 다시 실행하세요.'
os.chdir(repo)
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'

expected = {
    'transformers': '4.46.3',
    'tokenizers': '0.20.3',
    'peft': '0.13.2',
    'accelerate': '1.14.0',
    'deepspeed': '0.19.7',
    'huggingface-hub': '0.36.2',
    'numpy': '2.1.3',
}
for package, wanted in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == wanted, (package, installed, wanted)

print('torch:', torch.__version__)
print('GPU count:', torch.cuda.device_count())
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2


In [ ]:
# 400명 전원의 이미지를 공개 FIUBench dataset revision에서 복원합니다.
import json, shutil, zipfile
from collections import defaultdict
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
repo=Path('/kaggle/working/FIUBench_kaggle')
records=[json.loads(s) for s in (repo/'dataset/full.json').read_text().splitlines() if s.strip()][:400]
needed={r['image_path'] for r in records}
assert len(needed)==400
rev=HfApi().dataset_info('gray311/FIUBench').sha
archive=hf_hub_download('gray311/FIUBench','SFHQ.zip',repo_type='dataset',revision=rev,local_dir='/kaggle/working/fiu_download')
with zipfile.ZipFile(archive) as z:
    entries=defaultdict(list)
    for m in z.infolist():
        p=Path(m.filename)
        if not m.is_dir() and '__MACOSX' not in p.parts and not p.name.startswith('._'):
            entries[p.name].append(m)
    for image_path in needed:
        dest=(repo/image_path).resolve()
        assert dest.is_relative_to((repo/'dataset').resolve())
        hits=entries[dest.name]; assert len(hits)==1,(dest.name,len(hits))
        dest.parent.mkdir(parents=True,exist_ok=True)
        with z.open(hits[0]) as src,dest.open('wb') as dst: shutil.copyfileobj(src,dst)
        with Image.open(dest) as im: im.verify()
assert all((repo/p).is_file() for p in needed)
Path('/kaggle/working/fiu_ga_data_revision.txt').write_text(rev)
print('이미지 검증:',len(needed),'dataset revision:',rev)


In [ ]:
# Kaggle 런타임 forget.py에 메모리 호환 패치와 fast tokenizer를 명시합니다.
# GA loss와 data_module.py는 변경하지 않습니다.
import ast
from pathlib import Path

path = repo / "forget.py"
text = path.read_text()
original = text

old = "AutoTokenizer.from_pretrained(cfg.model_path)"
new = "AutoTokenizer.from_pretrained(cfg.model_path, use_fast=True)"
assert old in text or new in text
text = text.replace(old, new)

marker = "    if cfg.LoRA.r != 0:\n        config = LoraConfig("
block = """    if str(model_cfg.get("gradient_checkpointing", "false")).lower() == "true":
        model.config.use_cache = False
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
        logger.info("Gradient checkpointing enabled", main_process_only=True)

"""
if "model.gradient_checkpointing_enable()" not in text:
    assert text.count(marker) == 1
    text = text.replace(marker, block + marker, 1)

end = text.rfind("\nif __name__ == \"__main__\":")
assert end > 0
if "if cfg.save_model and accelerator.is_main_process:" not in text:
    start = text.rfind("\n    accelerator.end_training()", 0, end)
    assert start > 0
    replacement = """
    output_dir = cfg.save_dir
    accelerator.wait_for_everyone()
    if cfg.save_model and accelerator.is_main_process:
        os.makedirs(output_dir, exist_ok=True)
        unwrapped_model = accelerator.unwrap_model(model)
        if cfg.LoRA.r != 0:
            save_lora_weights(unwrapped_model, output_dir)
        else:
            unwrapped_model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
    accelerator.wait_for_everyone()
    accelerator.end_training()
"""
    text = text[:start] + replacement + text[end:]

ast.parse(text)
backup = path.with_suffix(".py.before_fast_kaggle_patch")
if not backup.exists():
    backup.write_text(original)
path.write_text(text)
assert "use_fast=True" in text
assert "use_fast=False" not in text
assert "model.enable_input_require_grads()" in text
print("FAST tokenizer + Kaggle memory/save patch complete")


In [ ]:
# 원본과 별도로 감사용 학습 스크립트를 만듭니다. git push가 필요하지 않습니다.
import ast, hashlib, difflib, subprocess, json, sys
from pathlib import Path
audit_source = '"""Single-step cosine scheduling and fail-fast audit for the isolated GA notebook.\n\nNo torch import: numerical/control-flow tests can run without a GPU environment.\nThe notebook embeds this file, so it does not depend on a future git push.\n"""\nimport ast\nimport json\nimport math\nfrom pathlib import Path\n\n\nclass SchedulerAudit:\n    def __init__(self, scheduler, optimizer, total_steps, warmup_steps, log_path):\n        self.scheduler = scheduler\n        self.raw = scheduler.scheduler\n        self.optimizer = optimizer\n        self.total = int(total_steps)\n        self.warmup = int(warmup_steps)\n        assert scheduler.step_with_optimizer is False\n        assert self.raw.last_epoch == 0, "Expected a fresh LambdaLR; do not resume weights here"\n        assert 0 < self.warmup < self.total\n        a, b = self.raw.optimizer.param_groups, optimizer.param_groups\n        assert len(a) == len(b) and all(x is y for x, y in zip(a, b)), \\\n            "Scheduler and actual DeepSpeed optimizer do not share LR parameter groups"\n        self.attempts = self.successes = self.skips = 0\n        self.base_lrs = list(self.raw.base_lrs)\n        self.path = Path(log_path)\n        assert not self.path.exists(), f"Existing audit log: {self.path}; use a new run directory"\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.check()\n\n    def expected_lrs(self, step):\n        if step < self.warmup:\n            factor = step / self.warmup\n        else:\n            progress = (step - self.warmup) / (self.total - self.warmup)\n            factor = 0.5 * (1 + math.cos(math.pi * progress))\n        return [lr * factor for lr in self.base_lrs]\n\n    def check(self):\n        assert self.raw.last_epoch == self.successes, \\\n            ("Scheduler index differs from successful updates", self.raw.last_epoch, self.successes)\n        assert self.successes <= self.total\n        actual = [float(g[\'lr\']) for g in self.optimizer.param_groups]\n        assert len(actual) == len(self.base_lrs)\n        for got, wanted in zip(actual, self.expected_lrs(self.successes)):\n            assert math.isclose(got, wanted, rel_tol=1e-7, abs_tol=1e-12), (got, wanted)\n        return actual\n\n    def finish_microstep(self, *, sync, skipped, lr_before, epoch, microbatch, loss):\n        # DeepSpeed already stepped its optimizer inside accelerator.backward().\n        self.check()  # Also detects an unexpected engine-owned scheduler step.\n        assert lr_before == [float(g[\'lr\']) for g in self.optimizer.param_groups]\n        if not sync:\n            return None\n        self.attempts += 1\n        assert self.attempts <= self.total\n        assert math.isfinite(loss), f"Non-finite GA loss: {loss}"\n        if skipped:\n            self.skips += 1\n        else:\n            self.scheduler.step()\n            self.successes += 1\n        after = self.check()\n        record = dict(epoch=int(epoch), microbatch=int(microbatch),\n                      attempt=self.attempts, successful_updates=self.successes,\n                      skipped_updates=self.skips, skipped=bool(skipped),\n                      scheduler_index=int(self.raw.last_epoch),\n                      lr_used=lr_before, lr_next=after, ga_loss=float(loss))\n        with self.path.open(\'a\') as f:\n            f.write(json.dumps(record, allow_nan=False) + \'\\n\')\n        return record\n\n\ndef patch_training_source(text):\n    """Patch exact anchors; fail on source drift instead of silently doing nothing."""\n    def replace(old, new):\n        nonlocal text\n        assert text.count(old) == 1, f"Expected exactly one patch anchor: {old!r}"\n        text = text.replace(old, new, 1)\n\n    replace(\'from accelerate import Accelerator, DistributedType\',\n            \'from accelerate import Accelerator, DistributedType\\nfrom ga_scheduler_audit import SchedulerAudit\')\n    replace(\'        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\\n\',\n            \'        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\\n\'\n            \'        step_scheduler_with_optimizer=False,\\n\')\n    anchor = \'    model, optimizer, torch_format_dataloader, lr_scheduler = accelerator.prepare(model, optimizer, torch_format_dataloader, lr_scheduler)\'\n    replace(anchor, anchor + \'\'\'\n    assert cfg.forget_loss == "ga" and not cfg.resume_from_checkpoint\n    assert cfg.lr_scheduler_type == "cosine"\n    assert accelerator.num_processes == 2\n    assert len(torch_format_dataloader) == 50, len(torch_format_dataloader)\n    assert max_train_steps == 32, max_train_steps\n    assert getattr(model, "lr_scheduler", None) is None, "DeepSpeed must not step a second scheduler"\n    schedule_audit = SchedulerAudit(\n        lr_scheduler, optimizer, max_train_steps,\n        round(cfg.warmup_ratio * max_train_steps),\n        Path(cfg.save_dir) / f"scheduler_rank{accelerator.process_index}.jsonl",\n    )\n    accelerator.print("SCHEDULER_BINDING_OK: one scheduler step per successful optimizer update")\n\'\'\')\n    replace(\'                accelerator.backward(loss)\',\n            \'                audit_lr_before = schedule_audit.check()\\n\'\n            \'                accelerator.backward(loss)\')\n    replace(\'                lr_scheduler.step()\', \'\'\'                audit_record = schedule_audit.finish_microstep(\n                    sync=accelerator.sync_gradients,\n                    skipped=bool(accelerator.optimizer_step_was_skipped) if accelerator.sync_gradients else False,\n                    lr_before=audit_lr_before, epoch=epoch, microbatch=step,\n                    loss=float(loss.detach().float().item()) if accelerator.sync_gradients else 0.0,\n                )\n                if audit_record is not None:\n                    assert audit_record["attempt"] == completed_steps + 1\n                    accelerator.print("SCHEDULER_AUDIT " + json.dumps(audit_record))\n                    if audit_record["attempt"] == 4:\n                        assert audit_record["successful_updates"] > 0, "No successful updates in first epoch"\n                        accelerator.print("FIRST_EPOCH_SCHEDULER_OK — continuing same run")\'\'\')\n    # Keep 8 epochs and the original attempt-based checkpoint numbering unchanged.\n    anchor = \'    output_dir = cfg.save_dir\\n    accelerator.wait_for_everyone()\'\n    replace(anchor, \'\'\'    assert schedule_audit.attempts == max_train_steps\n    assert schedule_audit.successes + schedule_audit.skips == max_train_steps\n    schedule_audit.check()\n    accelerator.print(f"SCHEDULER_FINAL attempts={schedule_audit.attempts} successful={schedule_audit.successes} skipped={schedule_audit.skips}")\n\'\'\' + anchor)\n    ast.parse(text)\n    return text\n'
data_before = hashlib.sha256((repo/'data_module.py').read_bytes()).hexdigest()
(repo/'ga_scheduler_audit.py').write_text(audit_source)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
sys.modules.pop('ga_scheduler_audit', None)
namespace = {}
exec(compile(audit_source, 'ga_scheduler_audit.py', 'exec'), namespace)
original = (repo/'forget.py').read_text()
patched = namespace['patch_training_source'](original)
ast.parse(patched)
(repo/'forget_ga_scheduler_audit.py').write_text(patched)
assert hashlib.sha256((repo/'data_module.py').read_bytes()).hexdigest() == data_before
Path('/kaggle/working/ga_scheduler_fix.patch').write_text(''.join(difflib.unified_diff(
    original.splitlines(True), patched.splitlines(True), fromfile='forget.py', tofile='forget_ga_scheduler_audit.py')))
Path('/kaggle/working/ga_scheduler_environment.txt').write_text(
    sys.version + '\n' + subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
manifest = {'repo_commit': subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip(),
            'data_module_sha256': data_before,
            'training_script_sha256': hashlib.sha256(patched.encode()).hexdigest()}
Path('/kaggle/working/ga_scheduler_manifest.json').write_text(json.dumps(manifest,indent=2))
print('SCHEDULER_PATCH_READY; data_module.py unchanged')


In [ ]:
# 실제 설치된 AcceleratedScheduler로 CPU 단위 검증 (모델 학습/다운로드 없음).
import tempfile, warnings, math
from types import SimpleNamespace
from accelerate.scheduler import AcceleratedScheduler
from transformers import get_scheduler
from ga_scheduler_audit import SchedulerAudit
with tempfile.TemporaryDirectory() as d:
    p = torch.nn.Parameter(torch.tensor(1.0))
    opt = torch.optim.AdamW([p], lr=2e-5)
    raw = get_scheduler('cosine', opt, num_warmup_steps=2, num_training_steps=32)
    wrapped = AcceleratedScheduler(raw, opt, step_with_optimizer=False, split_batches=False)
    audit = SchedulerAudit(wrapped, opt, 32, 2, Path(d)/'check.jsonl')
    for i in range(32):
        before = audit.check()
        for _ in range(15):
            assert audit.finish_microstep(sync=False, skipped=False, lr_before=before,
                epoch=0, microbatch=0, loss=0.0) is None
        skipped = i in (0, 5)  # overflow를 모의하여 scheduler가 멈추는지도 검증
        if not skipped:
            p.grad = torch.ones_like(p)
            opt.step()
            opt.zero_grad()
        audit.finish_microstep(sync=True, skipped=skipped, lr_before=before,
            epoch=0, microbatch=i, loss=-0.5)
    assert audit.successes == raw.last_epoch == 30
    assert audit.skips == 2 and audit.attempts == 32
print('CPU_SCHEDULER_PREFLIGHT_OK (GPU/DeepSpeed 연결은 학습 실행 중 추가 검사)')


In [ ]:
# GA Forget5 config와 T4×2 Accelerate config를 명시적으로 작성합니다.
(repo / 'config/forget_ga_forget5_fast_scheduler_fix.yaml').write_text("""defaults:
  - forget_lora
  - _self_
model_family: llava-phi
model_path: gray311/vlm_unlearning_ft_llava_phi_3_mini
attn_implementation: sdpa
data_path: ./dataset/full.json
split: forget5
forget_loss: ga
lr: 2e-5
batch_size: 4
gradient_accumulation_steps: 16
num_epochs: 8
seed: 233
lr_scheduler_type: cosine
warmup_ratio: 0.06
resume_from_checkpoint: ""
LoRA: {r: 128, alpha: 256, dropout: 0.05}
workers: 0
report_to: null
save_dir: /kaggle/working/models/llava_phi_GA_forget5_fast_scheduler_fix
save_model: true
save_steps: 4
""")
(repo / 'config/accelerate_kaggle_fast_t4x2.yaml').write_text("""compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 16
  gradient_clipping: 1.0
  offload_optimizer_device: cpu
  offload_param_device: cpu
  zero3_init_flag: false
  zero_stage: 2
distributed_type: DEEPSPEED
downcast_bf16: 'no'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
use_cpu: false
""")
print((repo / 'config/forget_ga_forget5_fast_scheduler_fix.yaml').read_text())


In [ ]:
# 실제 GA 학습 400문장의 fast-tokenizer input_ids와 labels를 전부 검증합니다.
import hashlib
import json
import sys
from transformers import AutoTokenizer

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from data_module import preprocess_v1
from utils import get_model_identifiers_from_yaml

model_path = 'gray311/vlm_unlearning_ft_llava_phi_3_mini'
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
assert tokenizer.is_fast
assert tokenizer.__class__.__name__ == 'LlamaTokenizerFast'

model_cfg = get_model_identifiers_from_yaml('llava-phi')
question_tag = model_cfg['question_start_tag']
answer_tag = model_cfg['answer_tag']
roles = [question_tag, answer_tag]

split = json.loads((repo / 'dataset/split.json').read_text())
rows = [json.loads(x) for x in (repo / 'dataset/full.json').read_text().splitlines() if x.strip()][:400]
forget_ids = set(split['forget5'])
samples = [qa for row in rows if row['unique_id'] in forget_ids for qa in row['qa_list']]
assert len(forget_ids) == 20 and len(samples) == 400

digest = hashlib.sha256()
assistant_id = tokenizer.convert_tokens_to_ids('<|assistant|>')
for qa in samples:
    question = qa['question'].capitalize()
    answer = qa['answer'].capitalize()
    conversation = question_tag + '<image>\n' + question + answer_tag + answer
    encoded = tokenizer(conversation, max_length=512, truncation=True, return_tensors='pt')
    labels = preprocess_v1(tokenizer, encoded['input_ids'], conversation, roles)
    supervised = labels[0][labels[0] != -100]
    assert supervised.numel() > 1
    assert int(supervised[0]) == assistant_id
    digest.update(encoded['input_ids'].numpy().tobytes())
    digest.update(labels.numpy().tobytes())

signature = digest.hexdigest()
(Path('/kaggle/working') / 'ga_fast_training_token_signature_512.txt').write_text(signature + '\n')
print('tokenizer:', tokenizer.__class__.__name__, 'is_fast:', tokenizer.is_fast)
print('forget identities:', len(forget_ids), 'forget QA:', len(samples))
print('assistant token id:', assistant_id)
print('400-sample input/label SHA256:', signature)
print('effective forget batch:', 4 * 2 * 16)
print('예정 update attempts:', 32)
print('FAST_GA_TRAINING_PREFLIGHT_OK')


In [ ]:
# 이 셀 한 번으로 8 epochs. 첫 epoch 검증 통과 후 자동으로 이어갑니다.
import os, sys, subprocess
from pathlib import Path
out = Path('/kaggle/working/models/llava_phi_GA_forget5_fast_scheduler_fix')
assert not out.exists() or not any(out.iterdir()), f'기존 결과 보호: {out}. 같은 학습 셀을 중복 실행하지 마세요.'
env = os.environ.copy()
env.update(HF_HOME='/kaggle/working/hf_cache', TOKENIZERS_PARALLELISM='false',
           HYDRA_FULL_ERROR='1', PYTHONUNBUFFERED='1',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
           CUDA_VISIBLE_DEVICES='0,1', DS_SKIP_CUDA_CHECK='1')
cmd = [sys.executable, '-m', 'accelerate.commands.launch',
       '--config_file', 'config/accelerate_kaggle_fast_t4x2.yaml',
       '--main_process_port', '2233', 'forget_ga_scheduler_audit.py',
       '--config-name', 'forget_ga_forget5_fast_scheduler_fix']
log_path = Path('/kaggle/working/ga_forget5_scheduler_fix_console.log')
with log_path.open('w') as log:
    process = subprocess.Popen(cmd, cwd=repo, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()
    log.write(f'\nTRAIN_PROCESS_EXIT_{return_code}\n')
assert return_code == 0, f'학습/검증 실패: exit={return_code}. 마지막 오류를 확인하세요.'
print('TRAIN_PROCESS_EXIT_0')


In [ ]:
# 두 GPU의 실제 scheduler 진행 기록과 체크포인트를 검증합니다.
import json, math
from pathlib import Path
import torch
out = Path('/kaggle/working/models/llava_phi_GA_forget5_fast_scheduler_fix')
a = [json.loads(s) for s in (out/'scheduler_rank0.jsonl').read_text().splitlines()]
b = [json.loads(s) for s in (out/'scheduler_rank1.jsonl').read_text().splitlines()]
assert len(a) == len(b) == 32
for x,y in zip(a,b):
    for key in ('attempt','successful_updates','skipped_updates','scheduler_index','lr_used','lr_next','skipped'):
        assert x[key] == y[key], (key,x,y)
for i,r in enumerate(a,1):
    assert r['attempt'] == i
    assert r['successful_updates'] + r['skipped_updates'] == i
    assert r['scheduler_index'] == r['successful_updates']
    if i > 1 and a[i-2]['scheduler_index'] >= 2:
        assert r['lr_next'][0] <= a[i-2]['lr_next'][0] + 1e-12
paths = [out/f'step_{n}/checkpoint.pt' for n in range(4,33,4)] + [out/'checkpoint.pt']
for path in paths:
    assert path.is_file(), path
    state = torch.load(path, map_location='cpu', weights_only=True)
    assert isinstance(state,dict) and len(state) == 452
    assert all(torch.isfinite(t).all().item() for t in state.values())
    del state
print('epoch | attempt | successful | skips | scheduler | LR used -> next')
for r in a:
    if r['microbatch'] == 49:
        print(f"{r['epoch']:5} | {r['attempt']:7} | {r['successful_updates']:10} | {r['skipped_updates']:5} | {r['scheduler_index']:9} | {r['lr_used'][0]:.6e} -> {r['lr_next'][0]:.6e}")
print('최종', a[-1])
print('SCHEDULER_AND_CHECKPOINT_AUDIT_OK')
print('이 표를 먼저 캡처하여 확인한 뒤, 동일한 FAST 평가로 이전 GA와 비교하세요.')


## 저장과 비교

`models/llava_phi_GA_forget5_fast_scheduler_fix` 전체(8개 중간 checkpoint, 최종 checkpoint,
`cfg.yaml`, `scheduler_rank0.jsonl`, `scheduler_rank1.jsonl`, `log.txt`)와
`ga_forget5_scheduler_fix_console.log`, `ga_scheduler_environment.txt`, `ga_scheduler_manifest.json`,
`ga_scheduler_fix.patch`, `ga_fast_training_token_signature_512.txt`, `fiu_ga_data_revision.txt`,
학습 스크립트가 들어 있는 `FIUBench_kaggle`을 보존하세요. 저장된 버전의 Output에서 확인한 뒤 세션을 종료하세요.

기존 `FIUBench_forget5_fast_eval_kaggle.ipynb`에서 `METHOD="GA"`로 평가합니다.
새 평가 노트북에 **이번 학습 Output만** 연결하여 checkpoint 후보가 하나가 되게 합니다.
이전 GA 결과와 같은 평가 코드를 사용해야 학습률 수정의 효과를 비교할 수 있습니다.
우선 Core의 Forget EM/ROUGE-L과 Retain ROUGE-L을 비교하세요. raw Truth Ratio 평균과 Min-K 평균을
논문의 가공된 Truth 점수나 MIA AUROC와 바로 비교하지 마세요.

이번 실험은 확인된 scheduler 문제 하나의 영향을 확인합니다. 저자의 early-stopping 시점이나
공개 초기 모델의 정확한 학습 이력까지 같음을 보장하지 않습니다.
